## Imports, prep dataset

In [1]:
import datasets
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch 


dataset = datasets.load_dataset("lapa-llm/fiftyfive-best")
#take only 10k samples for train and 1k for test
test_dataset = dataset['train'].select([i for i in list(range(10_000,11_000))])
dataset['train'] = dataset["train"].select([i for i in list(range(10_000))])

/home/nazara/UCU/NLP/ASSIGNMENT4/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
student_tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-270m-it")
student = AutoModelForCausalLM.from_pretrained("google/gemma-3-270m-it",
    dtype=torch.bfloat16,
    attn_implementation="sdpa",
    device_map="auto",)

teacher_tokenizer = AutoTokenizer.from_pretrained("INSAIT-Institute/MamayLM-Gemma-3-4B-IT-v1.0")
teacher = AutoModelForCausalLM.from_pretrained("INSAIT-Institute/MamayLM-Gemma-3-4B-IT-v1.0",
    dtype=torch.bfloat16,
    attn_implementation="sdpa",
    device_map="auto",)

Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.28it/s]


In [3]:
print(f"Student vocab (config): {student.config.get_text_config().vocab_size}")
print(f"Teacher vocab (config): {teacher.config.get_text_config().vocab_size}")
print(f"Student tokenizer length: {len(student_tokenizer)}")
print(f"Teacher tokenizer length: {len(teacher_tokenizer)}")

print(f"Student embeddings: {student.get_input_embeddings().weight.shape}")
print(f"Teacher embeddings: {teacher.get_input_embeddings().weight.shape}")

Student vocab (config): 262144
Teacher vocab (config): 262208
Student tokenizer length: 262145
Teacher tokenizer length: 262145
Student embeddings: torch.Size([262144, 640])
Teacher embeddings: torch.Size([262208, 2560])


In [4]:
vocab_size = student.config.get_text_config().vocab_size
teacher.config.get_text_config().vocab_size = vocab_size
teacher.resize_token_embeddings(vocab_size)

Gemma3TextScaledWordEmbedding(262144, 2560, padding_idx=0)

In [5]:
student_tokenizer.pad_token_id == teacher_tokenizer.pad_token_id 

True

In [6]:
teacher.config.pad_token_id == student.config.pad_token_id

False

In [7]:
teacher.config.pad_token_id = student.config.pad_token_id

In [8]:
teacher.config.pad_token_id == student.config.pad_token_id

True

In [9]:
messages = [
    {"role": "user", "content": "Хто такий Козак Мамай?"},
]

input_ids = teacher_tokenizer.apply_chat_template(
    messages,
      return_tensors="pt",
  add_generation_prompt=False,
  return_dict=True
)
teacher_tokenizer.decode(input_ids['input_ids'][0])

'<bos><start_of_turn>user\nХто такий Козак Мамай?<end_of_turn>\n'

In [10]:
output_ids = teacher.generate(
    input_ids=input_ids['input_ids'].to(teacher.device),
    attention_mask=input_ids['attention_mask'].to(teacher.device),
)

teacher_tokenizer.batch_decode(output_ids[:,len(input_ids['input_ids'][0]):], skip_special_tokens=True)

['## Козак Мамай: Легендарний герой української культури\n\nКозак Мамай']

In [11]:
prompt_template = """Translate the following text from English to Ukrainian.
If a phrase is ambiguous, choose the most contextually appropriate meaning.
Do not add anything extra, just provide the translation.
English text to translate: {en}
Ukrainian translation: """

def format_input(en, uk=None) -> list[dict[str, str]]:
    messages_prompt = [
        {"role": "user", "content": prompt_template.format(en=en)},
    ]
    if uk:
        messages_prompt.append({"role": "assistant", "content": uk})

    return messages_prompt


In [12]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(
    tokenizer=teacher_tokenizer,
    pad_to_multiple_of=8,
    padding='longest',
)

def preprocess_function(examples):
    input_ids = []
    labels = []

    for en, uk in zip(examples["en"], examples["uk"]):
        messages_prompt = format_input(en)

        messages_full = format_input(en, uk)

        prompt_ids = student_tokenizer.apply_chat_template(
            messages_prompt,
            tokenize=True,
            add_generation_prompt=True,  # inserts <start_of_turn>model
            truncation=True,
            max_length=512,
        )

        full_ids = student_tokenizer.apply_chat_template(
            messages_full,
            tokenize=True,
            add_generation_prompt=False,
            truncation=True,
            max_length=512,
        )

        label = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]

        input_ids.append(full_ids)
        labels.append(label)

    return {
        "input_ids": input_ids,
        "labels": labels,
    }

tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names,
)   


In [13]:
#largest number of tokens in training set

max_tokens = 0
for item in tokenized_datasets['train']:
    length = len(item['input_ids'])
    if length > max_tokens:
        max_tokens = length
print(f"Max tokens in training set: {max_tokens}")

Max tokens in training set: 193


In [14]:
#Example sample
print("Input IDs:", tokenized_datasets["train"][0]["input_ids"])
print("----")
print("Labels:", tokenized_datasets["train"][0]["labels"])

print("Decoded Input:", student_tokenizer.decode(tokenized_datasets["train"][0]["input_ids"]))
print("Loss part of decoded input:", student_tokenizer.decode([id for id in tokenized_datasets["train"][0]["labels"] if id != -100]))



Input IDs: [2, 105, 2364, 107, 40414, 506, 2269, 1816, 699, 5422, 531, 32774, 236761, 107, 2859, 496, 20412, 563, 64426, 236764, 5347, 506, 1346, 4403, 1755, 6780, 6590, 236761, 107, 6294, 711, 1138, 4658, 4481, 236764, 1164, 2847, 506, 13959, 236761, 107, 27832, 1816, 531, 17866, 236787, 38790, 532, 13623, 659, 1156, 1607, 56940, 236761, 107, 129282, 13959, 236787, 106, 107, 105, 4368, 107, 237941, 50414, 39074, 6444, 3266, 5637, 16268, 236866, 2192, 214067, 219372, 3691, 5992, 102561, 236761, 106, 107]
----
Labels: [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 237941, 50414, 39074, 6444, 3266, 5637, 16268, 236866, 2192, 214067, 219372, 3691, 5992, 102561, 2

In [15]:
batch = data_collator([tokenized_datasets['train'][0], tokenized_datasets['train'][1]])
print(batch['input_ids'].shape)  # (2, seq_len)
print(batch['labels'].shape)     # (2, seq_len)


torch.Size([2, 80])
torch.Size([2, 80])


In [16]:
import torch
from transformers import Trainer, TrainingArguments

class DistillationTrainer(Trainer):
    """
    Custom Trainer for Knowledge Distillation.
    Combines:
    1. Cross-Entropy loss (student predictions vs ground truth labels)
    2. KL-Divergence loss (student logits vs teacher logits)
    """
    
    def __init__(
        self,
        teacher_model,
        temperature: float = 2.0,
        ce_alpha: float = 2.0,
        kl_alpha: float = 5.0,
        *args,
        **kwargs
    ):
        super().__init__(*args, **kwargs)
        self.teacher_model = teacher_model
        self.teacher_model.eval()
        self.temperature = temperature
        self.ce_alpha = ce_alpha
        self.kl_alpha = kl_alpha
        
        if hasattr(self.model, 'device'):
            self.teacher_model.to(self.model.device)

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        # Get labels for CE loss
        labels = inputs.get("labels")
        
        # Student forward pass
        student_outputs = model(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            labels=labels,  # This computes CE loss internally
        )
        
        # Teacher forward pass (no gradients)
        with torch.no_grad():
            teacher_outputs = self.teacher_model(
                input_ids=inputs['input_ids'].to(self.teacher_model.device),
                attention_mask=inputs['attention_mask'].to(self.teacher_model.device),
            )
        
        # CE Loss (from student_outputs if labels provided, else compute manually)
        if labels is not None:
            ce_loss = student_outputs.loss
        else:
            ce_loss = torch.tensor(0.0, device=model.device)
        
        # KL Divergence Loss
        kl_loss_fct = torch.nn.KLDivLoss(reduction="batchmean")
        
        student_logits = student_outputs.logits
        teacher_logits = teacher_outputs.logits.to(student_logits.device)
        
        kl_loss = kl_loss_fct(
            torch.nn.functional.log_softmax(student_logits / self.temperature, dim=-1),
            torch.nn.functional.softmax(teacher_logits / self.temperature, dim=-1),
        ) * (self.temperature ** 2)
        
        # Combined loss
        loss = self.ce_alpha * ce_loss + self.kl_alpha * kl_loss
        
       
        return (loss, student_outputs) if return_outputs else loss

In [17]:
training_args = TrainingArguments(
    output_dir="./student_model",
    overwrite_output_dir=True,
    
    # Training duration
    num_train_epochs=5,  # 3-5 is typical for distillation
    
    # Batch size 
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # effective batch size = 4 * 4 = 16
    
    # Learning rate
    learning_rate=5e-5,  # 1e-5 to 5e-5 typical for distillation
    warmup_ratio=0.1,  # or warmup_steps=500
    lr_scheduler_type="cosine",
    
    # Optimization
    optim="adamw_torch",
    weight_decay=0.01,
    max_grad_norm=1.0,
    
    # Precision 
    bf16=True,  
    
    # Logging & saving
    logging_steps=50,
    save_steps=1000,
    save_total_limit=2,
    eval_steps=500,
)

trainer = DistillationTrainer(
    model=student,
    teacher_model=teacher,
    
    # Distillation hyperparameters
    temperature=2.0,
    ce_alpha=2.0,
    kl_alpha=5.0,
    
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    data_collator=data_collator,
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
The model is already on multiple devices. Skipping the move to device specified in `args`.


## Eval

In [18]:
from tqdm import tqdm
import sacrebleu

pad_token_id=student_tokenizer.pad_token_id
eos_token_id=student_tokenizer.eos_token_id

def eval_model(model,  test_subset, batch_size=16):
    model.eval()
    references = test_subset['uk']
    hypotheses = []

    test_examples = list(test_subset['en'])

    for i in tqdm(range(0, len(test_examples), batch_size)):
        batch_texts = test_examples[i:i+batch_size]
        
        messages_batch = [format_input(example) for example in batch_texts]

        input_ids = student_tokenizer.apply_chat_template(
            messages_batch,
            return_tensors="pt",
            add_generation_prompt=True,
            padding=True,
            return_dict=True
        )

        with torch.no_grad():
            output_ids = model.generate(
                input_ids=input_ids['input_ids'].to(model.device),
                attention_mask=input_ids['attention_mask'].to(model.device),
                max_new_tokens=128,
                do_sample=False,
                pad_token_id=pad_token_id,
                eos_token_id=eos_token_id,
                temperature=0.7,
                top_p=0.9,
            )

        decoded_outputs = student_tokenizer.batch_decode(output_ids, skip_special_tokens=True)
        
        for output in decoded_outputs:
            # Extract only the translation part
            translation = output.split("Ukrainian translation:\nmodel")[-1].strip()
            hypotheses.append(translation)

    bleu = sacrebleu.corpus_bleu(hypotheses, [references])
    chrf = sacrebleu.corpus_chrf(hypotheses, [references])
    result = {
        "bleu": bleu.score,
        "chrf": chrf.score,
        "hypotheses": hypotheses,
        "references": references
    }
    print("Evaluated BLEU:", bleu.score)
    print("Evaluated CHRF:", chrf.score)
    return result

In [19]:
res_student = eval_model(student, test_dataset, batch_size=16)

100%|██████████| 63/63 [02:09<00:00,  2.05s/it]

Evaluated BLEU: 0.5889521059523974
Evaluated CHRF: 2.006478074608896


In [20]:
res_teacher = eval_model(teacher, test_dataset, batch_size=16)

100%|██████████| 63/63 [03:58<00:00,  3.78s/it]

Evaluated BLEU: 49.69284589054025
Evaluated CHRF: 73.90248846830207


In [21]:
student.train()
trainer.train()

Step,Training Loss
50,10809.832500
100,5926.814400
150,4981.640900
200,4134.525900
250,2828.700000
300,2214.896900
350,1848.376600
400,1742.569200
450,1526.480500
500,1463.113700


TrainOutput(global_step=3125, training_loss=1286.397498125, metrics={'train_runtime': 1878.3119, 'train_samples_per_second': 26.62, 'train_steps_per_second': 1.664, 'total_flos': 2937834687406080.0, 'train_loss': 1286.397498125, 'epoch': 5.0})

In [22]:
res_student = eval_model(student, test_dataset, batch_size=16)

100%|██████████| 63/63 [02:09<00:00,  2.06s/it]

Evaluated BLEU: 38.020585255884676
Evaluated CHRF: 63.5678798513687


Baseline model :

Evaluated BLEU: 0.5889521059523974

Evaluated CHRF: 2.006478074608896

Baseline fine-tuned model (distilled):

Evaluated BLEU: 38.020585255884676

Evaluated CHRF: 63.5678798513687


Teacher model : 

Evaluated BLEU: 49.69284589054025

Evaluated CHRF: 73.90248846830207